# TP 11 – Le modèle de pourboire en temps réel, puis sous le capot (55 min)

**Objectif** : appliquer sur le flux de courses le modèle entraîné au TP 7, puis lire et comparer des plans d'exécution.

**Réussi si** : des prédictions sont produites en continu et vous savez expliquer le plan.

**Avant de commencer**, dans un terminal à la racine du dépôt :

```bash
python scripts/stream_simulator.py courses --reset          # en direct : un fichier toutes les 2 secondes
# ou, si vous êtes en retard / en mode dégradé :
python scripts/stream_simulator.py courses --all --reset    # tout d'un coup
```

In [ ]:
# Se placer à la racine du dépôt : les chemins data/... sont relatifs à la racine
import os
while not os.path.isdir("data") and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir("..")
print("Répertoire de travail :", os.getcwd())

In [ ]:
from pyspark.sql import SparkSession, functions as F
spark = SparkSession.builder.appName("TP").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")     # petit volume : 200 partitions par défaut, c'est trop

from pyspark.ml import PipelineModel
from pyspark.ml.functions import vector_to_array
import time
zones = spark.read.csv("data/taxi_zone_lookup.csv", header=True, inferSchema=True).select(
    F.col("LocationID").alias("PULocationID"), "Borough")

### 1. La préparation des features, dans UNE fonction réutilisable en batch et en streaming

Reprenez les étapes du TP 7 : carte uniquement, `heure`, jointure avec `zones` pour `Borough`, `manhattan`, `nuit`, `label` (pourboire > 0).

In [ ]:
def features(df):
    return (df.filter("trip_distance > 0 AND payment_type = 1")
              .withColumn("heure", F.hour("tpep_pickup_datetime"))
              .join(zones, "PULocationID")
              ...)

### 2. Recharger le modèle du TP 7 : `PipelineModel.load("out/modele_pourboire")`

S'il n'existe pas (ou si vous y avez sauvegardé le LinearSVC), réentraînez le Random Forest du TP 7 et sauvegardez-le à cet endroit : le TP a besoin de la colonne `probability`.

In [ ]:
modele = ...

### 3. Appliquer le modèle sur le flux

`modele.transform(features(flux))`, puis `p_pourboire` = `vector_to_array("probability")[1]`.

In [ ]:
schema = spark.read.parquet("data/trips.parquet").schema
flux = ...
predictions = ...

### 4. Taux de pourboire **prédit** (moyenne de `p_pourboire`) vs **réel** (moyenne de `label`), par fenêtre de 30 min et par `Borough`

Watermark de 10 minutes sur `ts` = `F.col("tpep_pickup_datetime").cast("timestamp")` : le watermark refuse le type `TIMESTAMP_NTZ` (sans fuseau) de nos données, d'où la conversion.

Pourquoi la moyenne des probabilités et pas des prédictions 0/1 ?

In [ ]:
suivi = ...

### 5. `foreachBatch` en `outputMode("append")` : pour chaque micro-batch, écrire en Parquet (`out/tp11/suivi`) et afficher le nombre de fenêtres

Le premier micro-batch écrit 0 fenêtre : aucune n'est encore close par le watermark. C'est normal.

In [ ]:
import shutil
for d in ["out/checkpoints/tp11", "out/tp11"]:
    shutil.rmtree(d, ignore_errors=True)

def traiter(batch_df, batch_id):
    ...

q = ...

### 6. Sous le capot : `explain("formatted")` sur la même requête, en CSV puis en Parquet

```python
requete = lambda df: df.filter("trip_distance > 10").groupBy("PULocationID").agg(F.avg("total_amount"))
```
Comparez `PushedFilters`, `ReadSchema` et `Batched`. Surprise : Catalyst pousse le filtre et élague les colonnes **dans les deux cas**. Qu'est-ce qui diffère, alors ? Mesurez le temps de chaque version (moyenne de 3 exécutions).

In [ ]:
...

### 7. AQE : le plan final n'existe qu'**après** l'exécution, sur **le même objet** DataFrame :

```python
r = requete(pq)
r.collect()
r.explain()
```
Trouvez `isFinalPlan=true` et `AQEShuffleRead coalesced`. Combien de blocs `*(n)` (whole-stage codegen) ? (Avec `requete(pq).explain()` vous obtiendriez un nouvel objet, encore `isFinalPlan=false`.)

In [ ]:
...

### Bonus – Spark UI, onglet **Structured Streaming** : débit, durée des micro-batches, lignes d'état